# Practice 1 — Internals, Typing, and Architecture

Data model, descriptors, generics, plugins, dependency injection.

**How it works:** each exercise is almost-finished code with a few `___` blanks. Fill them in, run the cell, then run the `check(...)` cell below it. ✅ means move on. ❌ shows your result and the test input, never the answer, and failing twice unlocks a hint.

Prefer a terminal? The same exercises are in [`practice/exercises/advanced_01_internals_typing_architecture.py`](../../practice/exercises/advanced_01_internals_typing_architecture.py).

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "exercises"))
    sys.path.insert(0, os.getcwd())

## Setup

Run this cell first.

In [ ]:
from __future__ import annotations
from check import Checker, ___  # ___ marks the blanks you fill in

check = Checker('advanced_01_internals_typing_architecture')

import inspect
from collections.abc import Callable, Iterable, Iterator, Sequence
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Generic, Optional, Protocol, TypeVar

T = TypeVar("T")

class UserRepository(Protocol):
    """Storage port: anything with get(user_id) and save(user) qualifies."""

    def get(self, user_id: str) -> Optional[dict]: ...
    def save(self, user: dict) -> None: ...

## Exercises

### 1. `ReadOnlyBag`

A read-only collection supporting len(), iteration, `in` and a safe repr.

In [ ]:
class ReadOnlyBag:
    """A read-only collection supporting len(), iteration, `in` and a safe repr.

    It copies its input so later changes to the caller's list do not leak in.
    repr shows at most 3 items: ReadOnlyBag([1, 2, 3, ...]) for longer bags.
    """

    def __init__(self, items: Iterable):
        self._items = ___

    def __len__(self) -> int:
        return ___

    def __iter__(self) -> Iterator:
        return ___

    def __contains__(self, item) -> bool:
        return ___

    def __repr__(self) -> str:
        shown = ", ".join(repr(i) for i in self._items[:3])
        more = ", ..." if len(self._items) > 3 else ""
        return ___

In [ ]:
check('ReadOnlyBag')

### 2. `Ticket`

Ordered by priority (lower first), then by sequence number.

In [ ]:
class Ticket:
    """Ordered by priority (lower first), then by sequence number.

    __eq__ and __lt__ return NotImplemented for non-Ticket operands, and
    Tickets are hashable on (priority, sequence).
    """

    def __init__(self, priority: int, sequence: int, title: str = ""):
        self.priority, self.sequence, self.title = ___

    def _key(self) -> tuple[int, int]:
        return ___

    def __eq__(self, other):
        if ___:
            return NotImplemented
        return self._key() == other._key()

    def __lt__(self, other):
        if ___:
            return NotImplemented
        return self._key() < other._key()

    def __hash__(self) -> int:
        return ___

In [ ]:
check('Ticket')

### 3. `PositiveInt`

Descriptor that only accepts positive ints (TypeError / ValueError otherwise).

In [ ]:
class PositiveInt:
    """Descriptor that only accepts positive ints (TypeError / ValueError otherwise).

    Usage:  class Order:  quantity = PositiveInt()
    Hint: __set_name__ learns the attribute name; store values in instance.__dict__.
    """

    def __set_name__(self, owner, name: str) -> None:
        self.name = ___

    def __get__(self, instance, owner=None):
        if ___:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value) -> None:
        if ___:
            raise TypeError(f"{self.name} must be an int")
        if ___:
            raise ValueError(f"{self.name} must be positive")
        instance.__dict__[self.name] = value

In [ ]:
check('PositiveInt')

### 4. `Stack`

A typed LIFO stack. pop() on an empty stack raises IndexError('pop from empty stack').

In [ ]:
class Stack(Generic[T]):
    """A typed LIFO stack. pop() on an empty stack raises IndexError('pop from empty stack')."""

    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(___)

    def pop(self) -> T:
        if ___:
            raise IndexError("pop from empty stack")
        return self._items.pop()

    def __len__(self) -> int:
        return ___

In [ ]:
check('Stack')

### 5. `first_or_none`

Return the first element or None, preserving the element type for type checkers.

In [ ]:
def first_or_none(values: Sequence[T]) -> Optional[T]:
    """Return the first element or None, preserving the element type for type checkers."""
    return ___

In [ ]:
check('first_or_none')

### 6. `PluginRegistry`

Run plugins only by approved name.

In [ ]:
class PluginRegistry:
    """Run plugins only by approved name.

    register(name, func) raises ValueError for names not in the allowlist;
    run(name, *args) raises KeyError for unregistered names.
    """

    def __init__(self, allowed: Iterable[str]):
        self._allowed = ___
        self._plugins: dict[str, Callable] = {}

    def register(self, name: str, func: Callable) -> None:
        if ___:
            raise ValueError(f"plugin {name!r} is not allowed")
        self._plugins[name] = func

    def run(self, name: str, *args):
        if ___:
            raise KeyError(name)
        return self._plugins[name](*args)

    def names(self) -> list[str]:
        return ___

In [ ]:
check('PluginRegistry')

### 7. `parameter_defaults`

Map each parameter name to its default value, or None if it has no default.

In [ ]:
def parameter_defaults(func: Callable) -> dict[str, object]:
    """Map each parameter name to its default value, or None if it has no default.

    Hint: inspect.signature(func).parameters and inspect.Parameter.empty.
    """
    return ___

In [ ]:
check('parameter_defaults')

### 8. `InMemoryUserRepository`

Satisfies UserRepository structurally (no inheritance needed).

*Checked together with a later exercise that uses it.*

In [ ]:
class InMemoryUserRepository:
    """Satisfies UserRepository structurally (no inheritance needed)."""

    def __init__(self) -> None:
        self._users: dict[str, dict] = {}

    def get(self, user_id: str) -> Optional[dict]:
        return ___

    def save(self, user: dict) -> None:
        self._users[user["id"]] = ___

### 9. `rename_user`

Domain service: validate, load, change, save. Raise KeyError if missing,

In [ ]:
def rename_user(repo: UserRepository, user_id: str, new_name: str) -> dict:
    """Domain service: validate, load, change, save. Raise KeyError if missing,
    ValueError if new_name is blank. Works with any UserRepository."""
    if ___:
        raise ValueError("name cannot be blank")
    user = repo.get(user_id)
    if ___:
        raise KeyError(user_id)
    updated = {**user, "name": new_name.strip()}
    repo.save(updated)
    return updated

In [ ]:
check('rename_user')

### 10. `Session`

A login session. is_expired() uses an injected clock so tests control time.

In [ ]:
@dataclass
class Session:
    """A login session. is_expired() uses an injected clock so tests control time."""
    created_at: datetime
    ttl: timedelta
    clock: Callable[[], datetime] = datetime.now

    def is_expired(self) -> bool:
        return ___

In [ ]:
check('Session')

### 11. `Vector`

2-D vector supporting +, -, * (by a number, either side), abs(), ==, and hashing.

In [ ]:
class Vector:
    """2-D vector supporting +, -, * (by a number, either side), abs(), ==, and hashing.

    repr is 'Vector(x, y)'. Unsupported operands return NotImplemented.
    """
    __slots__ = ("x", "y")

    def __init__(self, x: float, y: float):
        self.x, self.y = ___

    def __add__(self, other):
        if ___:
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        if ___:
            return NotImplemented
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, factor):
        if ___:
            return NotImplemented
        return Vector(self.x * factor, self.y * factor)

    __rmul__ = __mul__

    def __abs__(self) -> float:
        return ___

    def __eq__(self, other):
        if ___:
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self) -> int:
        return ___

    def __repr__(self) -> str:
        return ___

In [ ]:
check('Vector')

### 12. `LRUCache`

Fixed-capacity cache: get() marks a key as recently used; put() evicts the least

In [ ]:
class LRUCache:
    """Fixed-capacity cache: get() marks a key as recently used; put() evicts the least
    recently used key when full. get() of a missing key returns None. Hint: OrderedDict."""

    def __init__(self, capacity: int):
        from collections import OrderedDict
        if ___:
            raise ValueError("capacity must be positive")
        self.capacity = capacity
        self._data: OrderedDict = OrderedDict()

    def get(self, key):
        if ___:
            return None
        self._data.move_to_end(key)
        return ___

    def put(self, key, value) -> None:
        self._data[key] = value
        self._data.move_to_end(___)
        if ___:
            self._data.popitem(last=False)

    def keys(self) -> list:
        """Keys from least to most recently used."""
        return ___

In [ ]:
check('LRUCache')

### 13. `Plugin`

Base class whose subclasses register themselves by `name` via __init_subclass__.

In [ ]:
class Plugin:
    """Base class whose subclasses register themselves by `name` via __init_subclass__.

    Plugin.registry maps name -> subclass. A subclass without a `name` class attribute,
    or with a duplicate name, raises TypeError at class-creation time.
    """
    registry: dict[str, type] = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        name = cls.__dict__.get("name")
        if ___:
            raise TypeError(f"{cls.__name__} must define a string `name`")
        if ___:
            raise TypeError(f"duplicate plugin name {name!r}")
        Plugin.registry[name] = cls

In [ ]:
check('Plugin')

### 14. `enforce_types`

Decorator: raise TypeError when an argument is not an instance of its annotation.

In [ ]:
def enforce_types(func):
    """Decorator: raise TypeError when an argument is not an instance of its annotation.

    Only check parameters annotated with a plain class (int, str, ...); skip others.
    Hint: inspect.signature(func).bind(*args, **kwargs).
    """
    import functools
    signature = ___
    hints = ___

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        bound = signature.bind(*args, **kwargs)
        for name, value in bound.arguments.items():
            expected = hints.get(name)
            if ___:
                raise TypeError(f"{name} must be {expected.__name__}, got {type(value).__name__}")
        return ___
    return wrapper

In [ ]:
check('enforce_types')

## Finished?

In [ ]:
check.summary()